Summary: 

Import modules needed

In [1]:
import sdf_xarray as sdfxr
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
file_path = "/home/pnd531/Desktop/Project_EPOCH/dev_test/test_lbw/test_new_branch/*.sdf"
sdfxr.open_dataset("/home/pnd531/Desktop/Project_EPOCH/dev_test/test_lbw/test_new_branch/0020.sdf",keep_particles=True) # Check variables

<xarray.Dataset> Size: 37MB
Dimensions:                          (X_Grid_mid: 500, Y_Grid_mid: 500,
                                      ID_electron: 31969, ID_photon: 358049,
                                      ID_pos_bw: 2, ID_ele_bw: 2,
                                      ID_pos_lbw: 1, ID_ele_lbw: 1,
                                      X_Grid: 501, Y_Grid: 501)
Coordinates: (12/16)
  * X_Grid_mid                       (X_Grid_mid) float64 4kB -9.98e-06 ... 9...
  * Y_Grid_mid                       (Y_Grid_mid) float64 4kB -9.98e-06 ... 9...
    X_Particles_electron             (ID_electron) float64 256kB ...
    Y_Particles_electron             (ID_electron) float64 256kB ...
    X_Particles_photon               (ID_photon) float64 3MB ...
    Y_Particles_photon               (ID_photon) float64 3MB ...
    ...                               ...
    X_Particles_pos_lbw              (ID_pos_lbw) float64 8B ...
    Y_Particles_pos_lbw              (ID_pos_lbw) float64 8B ...
    X_Particles_ele_lbw              (ID_ele_lbw) float64 8B ...
    Y_Particles_ele_lbw              (ID_ele_lbw) float64 8B ...
  * X_Grid                           (X_Grid) float64 4kB -1e-05 ... 1e-05
  * Y_Grid                           (Y_Grid) float64 4kB -1e-05 ... 1e-05
Dimensions without coordinates: ID_electron, ID_photon, ID_pos_bw, ID_ele_bw,
                                ID_pos_lbw, ID_ele_lbw
Data variables: (12/36)
    Wall_time                        float64 8B ...
    Electric_Field_Ey                (X_Grid_mid, Y_Grid_mid) float32 1MB ...
    Electric_Field_Ey_averaged       (X_Grid_mid, Y_Grid_mid) float32 1MB ...
    Magnetic_Field_Bz                (X_Grid_mid, Y_Grid_mid) float32 1MB ...
    Magnetic_Field_Bz_averaged       (X_Grid_mid, Y_Grid_mid) float32 1MB ...
    Particles_Weight_electron        (ID_electron) float64 256kB ...
    ...                               ...
    Derived_Number_Density_electron  (X_Grid_mid, Y_Grid_mid) float64 2MB ...
    Derived_Number_Density_photon    (X_Grid_mid, Y_Grid_mid) float64 2MB ...
    Derived_Number_Density_pos_bw    (X_Grid_mid, Y_Grid_mid) float64 2MB ...
    Derived_Number_Density_ele_bw    (X_Grid_mid, Y_Grid_mid) float64 2MB ...
    Derived_Number_Density_pos_lbw   (X_Grid_mid, Y_Grid_mid) float64 2MB ...
    Derived_Number_Density_ele_lbw   (X_Grid_mid, Y_Grid_mid) float64 2MB ...
Attributes: (12/22)
    filename:         /home/pnd531/Desktop/Project_EPOCH/dev_test/test_lbw/te...
    file_version:     1
    file_revision:    4
    code_name:        Epoch2d
    step:             884
    time:             6.672161387797511e-14
    ...               ...
    compile_flags:    unknown
    defines:          50364608
    compile_date:     Wed Mar 25 15:36:11 2026
    run_date:         Wed Mar 25 15:38:16 2026
    io_date:          Wed Mar 25 15:39:24 2026
    deck:             {'control': {'nx': 500, 'ny': 500, 'x_min': -1e-05, 'x_...

Load the files

In [ ]:
ds= sdfxr.open_mfdataset(file_path)

Animation

In [ ]:
da = ds["Derived_Number_Density_pos_lbw"]
anim = da.epoch.animate()
anim.show()

Animate a lineout

In [ ]:
da = ds["Derived_Number_Density_pos_lbw"]
da_lineout = da.sel(Y_Grid_mid = 0.5e-6, method = "nearest")
anim = da_lineout.epoch.animate(title = "Y = 1e-6 [m]")
anim.show()

Let's animate a 3D simulation. Be very careful about the file size and the memory of the PC! Load only the variables you need to save memory. Open the activity monitor and look at the memory used.

In [ ]:
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Grab all the SDF files in your directory and sort them chronologically
file_path = "/home/pnd531/Desktop/Project_EPOCH/dev_test/test_lbw/sdf_files/*.sdf"
files = sorted(glob.glob(file_path))

times_fs = []
physical_electrons = []
macro_electrons = []

print("Scanning SDF files for LBW particles...")

for f in files:
    try:
        # Load individual file
        ds = xr.open_dataset(f,keep_particles=True)
        
        # Extract the time in femtoseconds
        t = ds.coords['Epoch'].values.item() if 'Epoch' in ds.coords else 0.0
        times_fs.append(t * 1e15)
        
        # The key for particle weights in sdf-xarray
        weight_key = "Particles_Weight_ele_lbw"
        
        if weight_key in ds.data_vars:
            weights = ds[weight_key].values
            
            # Sum of weights = actual physical number of electrons
            physical_electrons.append(np.sum(weights))
            
            # Length of array = number of computational macroparticles
            macro_electrons.append(len(weights))
        else:
            physical_electrons.append(0.0)
            macro_electrons.append(0)
            
        ds.close()
        
    except Exception as e:
        print(f"Error reading {f}: {e}")

# --- Plotting the Results ---
plt.figure(figsize=(12, 5))

# Plot 1: Physical Particle Count
plt.subplot(1, 2, 1)
plt.plot(times_fs, physical_electrons, marker='o', color='b', linestyle='-')
plt.title("Total Physical LBW Electrons")
plt.xlabel("Time (fs)")
plt.ylabel("Number of Physical Particles")
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Macroparticle Count
plt.subplot(1, 2, 2)
plt.plot(times_fs, macro_electrons, marker='x', color='r', linestyle='-')
plt.title("Total Macro LBW Electrons")
plt.xlabel("Time (fs)")
plt.ylabel("Macroparticles Tracked")
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

We can take a y-slice to look at a variable in the z-x plane.

In [ ]:
import glob
import numpy as np
import xarray as xr
import sdf_xarray as sdfxr



# Load times from the multifile dataset
try:
    ds_all = sdfxr.open_mfdataset(file_path)
    times = ds_all['time'].values
    ds_all.close()
except Exception as e:
    print(f"Error loading multifile dataset: {e}")
    times = []

files = sorted(glob.glob(file_path))

print("Scanning for Breit-Wheeler Positrons (Density and Particles)...")

for i, f in enumerate(files):
    if i >= len(times):
        break # Safety check in case of file mismatch
        
    try:
        ds = xr.open_dataset(f)
        t_fs = times[i] * 1e15 
        
        # 1. Check the 2D Density Grid
        if "Derived_Number_Density_pos_lbw" in ds.data_vars:
            # Extract the 400x400 array
            density_array = ds["Derived_Number_Density_pos_lbw"].values
            
            # Add up all the values in the array
            total_density_sum = np.sum(density_array)
            max_density = np.max(density_array)
            
            print(f"Time: {t_fs:5.1f} fs | Max Density: {max_density:.2e} m^-3 | Grid Sum: {total_density_sum:.2e}")
        else:
            print(f"Time: {t_fs:5.1f} fs | 'Derived_Number_Density_pos_lbw' DNE")

        # 2. Check the raw particle weights
        if "Particles_Weight_pos_lbw" in ds.data_vars:
            weights = ds["Particles_Weight_pos_lbw"].values
            physical_count = np.sum(weights)
            macro_count = len(weights)
            print(f"  -> Macro-Positrons: {macro_count:5d} | Physical Positrons: {physical_count:.2e}")
        else:
            print("  -> Variable 'Particles_Weight_pos_lbw' DNE (Zero particles)")
            
        ds.close()
    except Exception as e:
        print(f"Error on time slice {i}: {e}")

Check whether photon energies pass the LBW threshold

In [ ]:
import glob
import numpy as np
import xarray as xr
import sdf_xarray as sdfxr

# 1. Load the multifile dataset just like you did to get the clean time array
file_path = "/home/pnd531/Desktop/data_depository/linear_breit_wheeler/test_whether_it_works/sdf_files/tmp/*.sdf"
ds_all = sdfxr.open_mfdataset(file_path)
times = ds_all['time'].values
ds_all.close()

# 2. Grab the files to loop through for the ragged particle arrays
files = sorted(glob.glob(file_path))

print("Scanning for high-energy photons...")

for i, f in enumerate(files):
    try:
        # Load individual file for safe particle extraction
        ds = xr.open_dataset(f,keep_particles=True)
        
        # Use the time array we extracted via mfdataset
        t_fs = times[i] * 1e15 
        
        # Check if photons exist in this file
        if "Particles_Px_photon" in ds.data_vars:
            px = ds["Particles_Px_photon"].values
            py = ds["Particles_Py_photon"].values
            pz = ds["Particles_Pz_photon"].values
            
            # Momentum magnitude: p = sqrt(px^2 + py^2 + pz^2)
            p = np.sqrt(px**2 + py**2 + pz**2)
            
            # For photons, E = p * c
            energy_joules = p * 3e8
            energy_mev = energy_joules / (1.602e-13) 
            
            max_energy = np.max(energy_mev)
            num_photons = len(px) 
            
            print(f"Time: {t_fs:5.1f} fs | Photons: {num_photons:7d} | Max Energy: {max_energy:7.2f} MeV")
            
            if max_energy > 0.511:
                 print("  -> SUCCESS: Photons cross the 0.511 MeV rest-mass threshold!")
        else:
            print(f"Time: {t_fs:5.1f} fs | Photons: 0")
            
        ds.close()
    except Exception as e:
        print(f"Error reading particles from {f}: {e}")

In [ ]:
import gc 
gc.collect()